In [0]:
# Load the table from the catalog
health_irreg_rhythm = spark.read.table("workspace.default.health_irreg_rhythm")

In [0]:
health_irreg_rhythm.printSchema()

In [0]:
health_irreg_rhythm.show()

In [0]:
from pyspark.sql.functions import regexp_extract, to_date, when, col, median

# Parse Date_fixed
health_irreg_rhythm = health_irreg_rhythm.withColumn(
    "Date_fixed",
    when(regexp_extract(col("Date"), r"^[A-Za-z]{3}-[0-9]{1,2}-[0-9]{4}$", 0) != "",
         to_date(col("Date"), "MMM-d-yyyy"))
    .when(regexp_extract(col("Date"), r"^[0-9]{1,2}/[0-9]{1,2}/[0-9]{2}$", 0) != "",
         to_date(col("Date"), "M/d/yy"))
    .otherwise(None)
)

# Smoker_fixed
health_irreg_rhythm = health_irreg_rhythm.withColumn(
    "Smoker_fixed",
    when(col("Smoker") == "yes", True)
    .when(col("Smoker") == "no", False)
    .otherwise(None)
)

# Heart_Rate_fixed (replace 1600 with median)
valid_heart_rates = health_irreg_rhythm.filter(col("Heart Rate") != 1600).select("Heart Rate")
median_heart_rate = valid_heart_rates.approxQuantile("Heart Rate", [0.5], 0.001)[0]
health_irreg_rhythm = health_irreg_rhythm.withColumn(
    "Heart_Rate_fixed",
    when(col("Heart Rate") == 1600, median_heart_rate).otherwise(col("Heart Rate"))
)

# Rings_Closed_fixed
health_irreg_rhythm = health_irreg_rhythm.withColumn(
    "Rings_Closed_fixed",
    when(col("Rings Closed") == 4, 3).otherwise(col("Rings Closed"))
)

# Cohort_fixed
health_irreg_rhythm = health_irreg_rhythm.withColumn(
    "Cohort_fixed",
    when((col("Cohort") == 3) & (col("Id") == 3), 4).otherwise(col("Cohort"))
)

health_irreg_rhythm.show(5)


In [0]:
health_irreg_rhythm.printSchema()